# B7 — Adapter quantization test (`quant_int8`)

**Question:** does quantizing the 512x768 adapter matrix W to **int8**
(0.39 MB instead of 0.79 MB fp16) preserve retrieval ranking?

**Method:** symmetric per-output-channel int8 quantization of W
(scale = max|w|/127 per column), then the exact B3 retrieval protocol on
the held-out 1,000 images, comparing fp32 / fp16 / int8 adapters.

**Pass criteria:** (1) per-image cosine between fp32-adapted and
int8-adapted vectors > 0.999; (2) every Recall@K within 1 point absolute
of the fp16 adapter. Runtime: seconds; needs only `pairs.npz` +
`adapter.npz`.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])

def l2n(X):
    return X / (np.linalg.norm(X, axis=-1, keepdims=True) + 1e-9)

def recall(sim, ks=(1, 5, 10)):
    ranks = (-sim).argsort(axis=1)
    n = sim.shape[0]
    return {k: float((ranks[:, :k] == np.arange(n)[:, None]).any(1).mean())
            for k in ks}

def report(name, r, ceil=None):
    line = f"{name:<38} " + "  ".join(f"R@{k}={r[k]:.3f}" for k in (1, 5, 10))
    if ceil:
        pct = min(100 * r[k] / max(ceil[k], 1e-9) for k in (1, 5, 10))
        line += f"   (worst-K {pct:.1f}% of ref)"
    print(line)

pairs = np.load(DATA_DIR / 'pairs.npz')
ad = np.load(DATA_DIR / 'adapter.npz')
te = ad['eval_idx']
W = ad['W_ridge'].astype(np.float32)

# --- quantizers ---
def quant_int8_per_col(W):
    scale = np.abs(W).max(axis=0, keepdims=True) / 127.0
    Wq = np.round(W / scale).clip(-127, 127).astype(np.int8)
    return Wq, scale

Wq, scale = quant_int8_per_col(W)
W_int8 = Wq.astype(np.float32) * scale          # dequantized view
W_fp16 = W.astype(np.float16).astype(np.float32)
print(f"int8 payload: {Wq.nbytes/1e6:.2f} MB + {scale.nbytes/1e3:.1f} KB scales "
      f"(fp16 was {W.astype(np.float16).nbytes/1e6:.2f} MB)")

In [ ]:
# Parity: per-image cosine of adapted vectors, fp32 vs fp16 vs int8
x = pairs['mob_img'][te]
v32, v16, v8 = l2n(x @ W), l2n(x @ W_fp16), l2n(x @ W_int8)
print('fp16 vs fp32  min cosine:', float((v32*v16).sum(1).min()))
c8 = (v32*v8).sum(1)
print('int8 vs fp32  min cosine:', float(c8.min()),
      ' mean:', float(c8.mean()))
print('PASS parity' if c8.min() > 0.999 else 'FAIL parity')

In [ ]:
# Retrieval: exact B3 protocol, three adapter precisions
txt, sig = pairs['sig_txt'][te], pairs['sig_img'][te]
ceil = recall(txt @ sig.T)
r32, r16, r8 = (recall(txt @ v.T) for v in (v32, v16, v8))
report('SigLIP native (ceiling)', ceil)
report('adapter fp32', r32, ceil)
report('adapter fp16', r16, ceil)
report('adapter int8', r8, ceil)
ok = all(abs(r8[k]-r16[k]) <= 0.01 for k in (1,5,10))
print('\nPASS: int8 within 1pt of fp16 at every K' if ok
      else 'FAIL: int8 shifted recall by >1pt - keep fp16')

**Interpretation guide.** fp16 was previously verified ranking-neutral
(min cosine 0.9999999). int8 has only ~255 levels per column; expect min
cosine around 0.999x and identical or near-identical recall. If it
passes: the adapter *can* ship at 0.39 MB - though the saving is
negligible next to the ~30 MB encoder, so fp16 remains the sensible
default. If it fails: nothing is lost; this test is exactly why the
decision is empirical.